# Lab 02 Extra - Banco de Dados Northwind
**Disciplina:** Extração e Preparação de Dados | **Professor:** Luis Aramis

Este é um notebook extra para praticar SQL com um banco de dados clássico: o **Northwind**.
Ele simula uma importadora/exportadora de alimentos gourmet.

## 1. Setup e Download
Vamos baixar o `northwind.db` e conectar o SQLAlchemy.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
import urllib.request

# # Download do northwind.db
# if not os.path.exists('northwind.db'):
#     # URL do repositório jpwhite3/northwind-SQLite3
#     url = 'https://github.com/jpwhite3/northwind-SQLite3/raw/main/dist/northwind.db'
#     urllib.request.urlretrieve(url, 'northwind.db')
#     print('Banco Northwind baixado com sucesso!')

engine = create_engine('sqlite:///northwind.db')
print('Conexão estabelecida!')

Conexão estabelecida!


## 2. Mapa do Banco
Quais tabelas temos aqui?


In [2]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, engine)
print(tables)


                    name
0             Categories
1        sqlite_sequence
2   CustomerCustomerDemo
3   CustomerDemographics
4              Customers
5              Employees
6    EmployeeTerritories
7          Order Details
8                 Orders
9               Products
10               Regions
11              Shippers
12             Suppliers
13           Territories


## 3. Consultas Básicas
1. Liste os 5 produtos mais caros (`Products`).
2. Liste todos os clientes (`Customers`) que moram no 'Brazil'.

In [3]:
# Produtos mais caros
query_produtos_caros = """
SELECT ProductName, UnitPrice
FROM Products
ORDER BY UnitPrice DESC
LIMIT 5;
"""
df_produtos_caros = pd.read_sql(query_produtos_caros, engine)
print("Top 5 produtos mais caros:")
print(df_produtos_caros)


Top 5 produtos mais caros:
               ProductName  UnitPrice
0            Côte de Blaye     263.50
1  Thüringer Rostbratwurst     123.79
2          Mishi Kobe Niku      97.00
3   Sir Rodney's Marmalade      81.00
4         Carnarvon Tigers      62.50


In [4]:
# Clientes do Brasil
query_clientes_brasil = """
SELECT CustomerID, CompanyName, ContactName, City, Country
FROM Customers
WHERE Country = 'Brazil';
"""
df_clientes_brasil = pd.read_sql(query_clientes_brasil, engine)
print("Clientes do Brasil:")
print(df_clientes_brasil)


Clientes do Brasil:
  CustomerID             CompanyName        ContactName            City  \
0      COMMI        Comércio Mineiro       Pedro Afonso       Sao Paulo   
1      FAMIA      Familia Arquibaldo          Aria Cruz       Sao Paulo   
2      GOURL     Gourmet Lanchonetes      André Fonseca        Campinas   
3      HANAR           Hanari Carnes       Mario Pontes  Rio de Janeiro   
4      QUEDE             Que Delícia   Bernardo Batista  Rio de Janeiro   
5      QUEEN           Queen Cozinha     Lúcia Carvalho       Sao Paulo   
6      RICAR      Ricardo Adocicados     Janete Limeira  Rio de Janeiro   
7      TRADH  Tradição Hipermercados  Anabela Domingues       Sao Paulo   
8      WELLI  Wellington Importadora      Paula Parente         Resende   

  Country  
0  Brazil  
1  Brazil  
2  Brazil  
3  Brazil  
4  Brazil  
5  Brazil  
6  Brazil  
7  Brazil  
8  Brazil  


## 4. JOIN: Pedidos e Clientes
Vamos ver quem fez quais pedidos.
Tabelas: `Orders` e `Customers`.
Chave de ligação: `CustomerID`.

In [5]:
query = """
SELECT o.OrderID, o.OrderDate, c.CustomerID, c.CompanyName, c.ContactName, c.Country
FROM Orders o
JOIN Customers c ON o.CustomerID = c.CustomerID
ORDER BY o.OrderDate ASC
LIMIT 10;
"""
df_orders_customers = pd.read_sql(query, engine)
print(df_orders_customers)


   OrderID            OrderDate CustomerID                   CompanyName  \
0    18429  2012-07-10 15:40:46      GREAL       Great Lakes Food Market   
1    25506  2012-07-10 20:28:57      RICAR            Ricardo Adocicados   
2    26048  2012-07-11 01:09:16      LONEP      Lonesome Pine Restaurant   
3    16958  2012-07-11 20:26:28      FOLKO                Folk och fä HB   
4    25877  2012-07-11 21:17:36      NORTS                   North/South   
5    16993  2012-07-11 21:17:46      AROUT               Around the Horn   
6    19158  2012-07-12 05:29:34      RATTC    Rattlesnake Canyon Grocery   
7    19258  2012-07-12 08:42:56      HUNGO  Hungry Owl All-Night Grocers   
8    14262  2012-07-12 11:39:39      RANCH                 Rancho grande   
9    25939  2012-07-12 12:10:41      SIMOB                 Simons bistro   

        ContactName    Country  
0     Howard Snyder        USA  
1    Janete Limeira     Brazil  
2       Fran Wilson        USA  
3     Maria Larsson     Sweden 

## 5. JOIN Triplo: Detalhes do Pedido
O que tem dentro do pedido 10248?
Caminho: `OrderDetails` -> `Products`.

In [9]:
query = """
SELECT od.OrderID, p.ProductName, od.Quantity, od.UnitPrice
FROM OrderDetails od
JOIN Products p ON od.ProductID = p.ProductID
WHERE od.OrderID = 10248;
"""
df_pedido = pd.read_sql(query, engine)
print(df_pedido)


DatabaseError: Execution failed on sql '
SELECT od.OrderID, p.ProductName, od.Quantity, od.UnitPrice
FROM OrderDetails od
JOIN Products p ON od.ProductID = p.ProductID
WHERE od.OrderID = 10248;
': (sqlite3.OperationalError) no such table: OrderDetails
[SQL: 
SELECT od.OrderID, p.ProductName, od.Quantity, od.UnitPrice
FROM OrderDetails od
JOIN Products p ON od.ProductID = p.ProductID
WHERE od.OrderID = 10248;
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 6. Desafio: Total de Vendas por Categoria
Descubra qual Categoria de produtos (`Categories`) gerou mais receita.
Dica: Você vai precisar ligar `Categories` -> `Products` -> `Order Details`.

In [10]:
# Seu código aqui
query = """
SELECT c.CategoryName, SUM(od.Quantity * od.UnitPrice) AS TotalRevenue
FROM OrderDetails od
JOIN Products p ON od.ProductID = p.ProductID
JOIN Categories c ON p.CategoryID = c.CategoryID
GROUP BY c.CategoryName
ORDER BY TotalRevenue DESC;
"""
df_revenue = pd.read_sql(query, engine)
print(df_revenue)


DatabaseError: Execution failed on sql '
SELECT c.CategoryName, SUM(od.Quantity * od.UnitPrice) AS TotalRevenue
FROM OrderDetails od
JOIN Products p ON od.ProductID = p.ProductID
JOIN Categories c ON p.CategoryID = c.CategoryID
GROUP BY c.CategoryName
ORDER BY TotalRevenue DESC;
': (sqlite3.OperationalError) no such table: OrderDetails
[SQL: 
SELECT c.CategoryName, SUM(od.Quantity * od.UnitPrice) AS TotalRevenue
FROM OrderDetails od
JOIN Products p ON od.ProductID = p.ProductID
JOIN Categories c ON p.CategoryID = c.CategoryID
GROUP BY c.CategoryName
ORDER BY TotalRevenue DESC;
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)